In [29]:
import kagglehub
import pandas as pd
import numpy as np
import os

try:
    path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
    data = pd.read_csv(os.path.join(path, 'Monday-WorkingHours.pcap_ISCX.csv'))
    print(" Path to dataset files:", path)
    
except Exception as e:
    print(f"Something failed... {e}")

data.head()

 Path to dataset files: /Users/antoniogonzalez/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [30]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data.columns = data.columns.str.strip()

data['Label'] = le.fit_transform(data['Label'])

data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

X = data.drop('Label', axis=1)
y = data['Label']

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=10)

In [32]:
from  xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)


''' 
    This is alot to unpack.

    enumerate is doing 3 things in this code, the fold counter itself,
    the two index arrays itself (number and row indices).

    skf.split is diving the code 5 times shuffling the data to be different
    at each fold.as_integer_ratio
    
    1 at the end being the iteration telling enumerate where to start

    x.iloc selects a row from x_train and x.test,
    x_train being the study guide test
    x_test being the actual test

    y_train: study guide answers
    y_test: actual test answers
'''

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx] 

#  mlogloss: confidence of the models proability predictions (multiclass)
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=10)
    xgb.fit(X_train, y_train)


    y_pred = xgb.predict(X_test)
    print('-Fold--')
    print(classification_report(y_test, y_pred))

# Noticeablly faster than RFC

-Fold--
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105897

    accuracy                           1.00    105897
   macro avg       1.00      1.00      1.00    105897
weighted avg       1.00      1.00      1.00    105897

-Fold--
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105896

    accuracy                           1.00    105896
   macro avg       1.00      1.00      1.00    105896
weighted avg       1.00      1.00      1.00    105896

-Fold--
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105896

    accuracy                           1.00    105896
   macro avg       1.00      1.00      1.00    105896
weighted avg       1.00      1.00      1.00    105896

-Fold--
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105896

    accuracy                           